In [89]:
#Aufgabe 2_1
#Wir haben die Tabelle aus Aufgabe 1 übernommen.
#Zunächst wandeln wir zur Sicherheit alle 0 Werte in NA um, daraufhin droppen wir diese Zeilen.
import pandas as pd
import numpy as np
listings_2 = pd.read_csv("listings_1_13.csv", index_col="Unnamed: 0")
listings_2["price"] = listings_2["price"].replace(0, pd.NA)
listings_2.dropna(subset=["price"], inplace=True)

print(listings_2.columns)


listings_2_1 = listings_2[["price", "accommodates","amenities_length","latitude","longitude", "bedrooms", "host_is_superhost", "Shared_bath_status", "instant_bookable", "bathrooms_number"]]
#Drops NA
listings_2_1.dropna(inplace=True)
listings_2_1["price"] = listings_2_1["price"].apply(np.log)
listings_2_1.rename(columns={"price":"log_price"}, inplace=True)
#We have checked that there are no NA values.  No further cleaning is necessary.

#print(len(listings_2["price"]))
#Ziel: Price
#Chosen Predictors: "region_median_price","latitude","longitude", "accommodates", "bedrooms", "host_is_superhost", "Shared_bath_status", instant_bookable", "bathrooms_number"
#listings_2_1.head()
#print(listings_2_1.isna().sum())



Index(['price', 'neighbourhood_cleansed', 'latitude', 'longitude', 'room_type',
       'bedrooms', 'bathrooms_number', 'accommodates', 'amenities',
       'host_is_superhost', 'host_listings_count', 'host_response_rate',
       'review_scores_rating', 'review_scores_cleanliness',
       'number_of_reviews', 'reviews_per_month', 'minimum_nights',
       'availability_365', 'instant_bookable', 'Shared_bath_status',
       'host_tenure_years', 'amenities_length', 'region_geo',
       'region_median_price', 'above_local_median'],
      dtype='str')


Based upon the results of the previous task, we were able to see some of the chosen correlating parameters. We further added some of the parameters that were not included in the previous task, such as the number of bedrooms and bathrooms, because we believe they have a significant impact on the price of the house.

In [ ]:
#Wir haben die Variablen latitude und longitude in eine Distanzvariable zum Stadtzentrum transformiert. Der Grund dafür ist, dass der Einfluss der Lage auf den Preis plausibel nicht linear entlang einer einzelnen geografischen Achse verläuft. Stattdessen ist zu erwarten, dass sich der Preiseffekt radial vom Stadtzentrum aus verändert, sodass vor allem die Entfernung zum Zentrum – und nicht die absolute Position in Nord-Süd- oder Ost-West-Richtung – für die Preisbildung relevant ist.
#Add distance from dataset geographic center using Euclidean distance.
   
lon_center = listings_2_1["longitude"].mean()
lat_center = listings_2_1["latitude"].mean()

listings_2_1["distance_center"] = np.sqrt(
    (listings_2_1["latitude"] - lat_center) ** 2 +
    (listings_2_1["longitude"] - lon_center) ** 2
)
  
listings_2_1 = listings_2_1.drop(columns=["latitude", "longitude"])

In [91]:
#Aufgabe_2_2
import statsmodels.api as sm
outcome = listings_2_1["log_price"]
regressor = listings_2_1.drop(columns=["log_price"])
regressor = sm.add_constant(regressor)

bool_cols = regressor.select_dtypes("bool").columns
regressor[bool_cols] = regressor[bool_cols].astype(int)
#print(regressor.dtypes)
olssim = sm.OLS(outcome, regressor)
olssim = olssim.fit()
olssim.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              log_price   R-squared:                       0.416
Model:                            OLS   Adj. R-squared:                  0.415
Method:                 Least Squares   F-statistic:                     818.6
Date:                Fri, 13 Mar 2026   Prob (F-statistic):               0.00
Time:                        15:14:35   Log-Likelihood:                -6681.9
No. Observations:                9203   AIC:                         1.338e+04
Df Residuals:                    9194   BIC:                         1.345e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
======================================================================================
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                  4.0557      0.018    221.642      0.000       4.020       4.092
accommodates           0.0864      0.004     23.871      0.000       0.079       0.094
amenities_length       0.0003   1.85e-05     15.124      0.000       0.000       0.000
bedrooms               0.1107      0.009     12.300      0.000       0.093       0.128
host_is_superhost      0.0336      0.012      2.877      0.004       0.011       0.056
Shared_bath_status    -0.4608      0.015    -30.791      0.000      -0.490      -0.431
instant_bookable       0.2201      0.011     19.782      0.000       0.198       0.242
bathrooms_number       0.1024      0.012      8.329      0.000       0.078       0.127
distance_center       -1.4169      0.106    -13.318      0.000      -1.625      -1.208
==============================================================================
Omnibus:                     4536.216   Durbin-Watson:                   1.607
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           124077.329
Skew:                           1.799   Prob(JB):                         0.00
Kurtosis:                      20.625   Cond. No.                     1.26e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.26e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [ ]:
#Aufgabe_2_3
